# GitHubからGoogleドライブへ一度だけ移行する

コード・画像・音声・データ・資料・起動ノートブックをまとめてコピーします。Gitの履歴や仮想環境は含みません。GitHub側のファイルは削除しません。

1. 最初のセルでドライブへの接続を許可します。
2. 次のセルを実行し、「移行完了」が出るまで待ちます。画像が多いため時間がかかります。
3. Googleドライブで **kinoko_quiz／kinoko-quiz／colab／launch_kinoko_quiz.ipynb** を探し、Google Colaboratoryで開きます。
4. 次回からはその起動ノートブックだけを使います。GitHubへの接続は不要です。

既存の移行先は上書きしません。途中で止まった場合は、同じセルを再実行できます。ドライブの空き容量は少なくとも1GBを用意してください。


In [ ]:
from google.colab import drive
from pathlib import Path

drive.mount('/content/drive')
DRIVE_PROJECT_DIR = Path('/content/drive/MyDrive/kinoko_quiz/kinoko-quiz')
# 配布ZIPをDriveにアップロード済みなら、そのパスを指定できます。
# 例: '/content/drive/MyDrive/kinoko_quiz/kinoko-quiz-drive.zip'
# 空欄なら、今回だけGitHubのmainを取得します。
ARCHIVE_PATH = ''
if DRIVE_PROJECT_DIR.exists():
    raise FileExistsError(f'移行先は既に存在します: {DRIVE_PROJECT_DIR}。移行済みなら起動ノートブックを開いてください。')


In [ ]:
import importlib.util
import shutil
import stat
import tempfile
import urllib.request
import zipfile
from pathlib import PurePosixPath

if DRIVE_PROJECT_DIR.exists():
    raise FileExistsError(f'既存の移行先は上書きしません: {DRIVE_PROJECT_DIR}')
with tempfile.TemporaryDirectory(prefix='quiz-migration-', dir='/content') as temporary:
    work = Path(temporary)
    if ARCHIVE_PATH:
        archive = Path(ARCHIVE_PATH)
        origin = 'Googleドライブの配布ZIP'
    else:
        origin = 'https://github.com/no-kishimoto/kinoko-quiz/tree/main'
        archive = work / 'project.zip'
        print('初回のみ：GitHubからコードと画像を取得します。', flush=True)
        url = 'https://codeload.github.com/no-kishimoto/kinoko-quiz/zip/refs/heads/main'
        with urllib.request.urlopen(url, timeout=120) as response, archive.open('wb') as output:
            downloaded = 0
            while True:
                chunk = response.read(8 * 1024 * 1024)
                if not chunk:
                    break
                output.write(chunk)
                downloaded += len(chunk)
                print(f'取得済み: {downloaded / 1024 / 1024:.0f} MB', flush=True)
    extracted = work / 'extracted'
    with zipfile.ZipFile(archive) as bundle:
        for member in bundle.infolist():
            path = PurePosixPath(member.filename)
            if path.is_absolute() or '..' in path.parts or '\\' in member.filename or stat.S_ISLNK(member.external_attr >> 16):
                raise ValueError('ZIP内のファイルのパスが不正です。')
        bundle.extractall(extracted)
    candidates = [p.parent for p in extracted.rglob('app.py')
                  if (p.parent / 'colab' / 'drive_files.py').is_file()]
    if len(candidates) != 1:
        raise ValueError('配布ZIPからゲームのフォルダを特定できませんでした。')
    source = candidates[0]
    helper = source / 'colab' / 'drive_files.py'
    spec = importlib.util.spec_from_file_location('quiz_drive_files', helper)
    drive_files = importlib.util.module_from_spec(spec)
    spec.loader.exec_module(drive_files)
    drive_files.migrate_project(source, DRIVE_PROJECT_DIR, origin)
print('ドライブ内の colab/launch_kinoko_quiz.ipynb を開いて起動してください。', flush=True)
